# Piste 1 — GAT topologique sur disconnectome (pipeline complet)

Ce notebook fait **tout automatiquement** :
1. Monte Google Drive
2. Installe les dépendances
3. Enrichit les graphes avec des **features topologiques** (betweenness, eigenvector, clustering, pagerank)
4. Définit et entraîne le **GAT topologique** (features de nœuds = 6)
5. Compare par **test de DeLong** : la topologie apporte-t-elle enfin quelque chose ?

**Tu n'as rien à modifier.** Exécute les cellules dans l'ordre (ou menu *Exécution → Tout exécuter*).

> Prérequis : le fichier `pyg_dataset.pt` doit déjà exister dans
> `isles_prp/MNI/nemo_results/` sur ton Drive (produit lors des sessions précédentes).


## 1. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Installer les dépendances et vérifier le GPU
(networkx est déjà présent ; on installe torch_geometric.)

In [ ]:
!pip -q install torch_geometric
import torch
print("PyTorch:", torch.__version__)
print("GPU dispo:", torch.cuda.is_available(), "|", 
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU seulement")
print("Astuce: si GPU=False -> menu Exécution > Modifier le type d'exécution > GPU (T4)")

## 3. Chemins (aucune modification nécessaire)

In [ ]:
import os
BASE   = "/content/drive/MyDrive/isles_prp/MNI"
NEMO   = os.path.join(BASE, "nemo_results")
PT_IN  = os.path.join(NEMO, "pyg_dataset.pt")            # dataset existant (x = 2 features)
PT_TOPO= os.path.join(NEMO, "pyg_dataset_topo.pt")       # sera créé (x = 6 features)
OUT_CSV= os.path.join(NEMO, "delong_topo.csv")

assert os.path.exists(PT_IN), f"Introuvable: {PT_IN} — vérifie que le Drive est monté et le fichier présent."
print("OK, dataset trouvé:", PT_IN)

## 4. Enrichissement topologique des graphes

On ajoute 4 mesures **structurelles** par région (issues de la forme du graphe, pas de l'intensité ChaCo).
Résultat : features de nœuds passent de 2 à **6**.

In [ ]:
import numpy as np, torch, networkx as nx

def node_topo_features(edge_index, edge_attr, n_nodes):
    G = nx.Graph(); G.add_nodes_from(range(n_nodes))
    if edge_index.numel() > 0:
        src, dst = edge_index[0].numpy(), edge_index[1].numpy()
        w = edge_attr.squeeze(-1).numpy() if edge_attr is not None else np.ones(len(src))
        for s,d,wt in zip(src,dst,w):
            if s < d:
                G.add_edge(int(s), int(d), weight=float(wt))
    for u,v,dd in G.edges(data=True):
        dd["dist"] = 1.0/(dd["weight"]+1e-6)
    try:    btw = nx.betweenness_centrality(G, weight="dist", normalized=True)
    except Exception: btw = {i:0.0 for i in range(n_nodes)}
    try:    eig = nx.eigenvector_centrality_numpy(G, weight="weight")
    except Exception: eig = {i:0.0 for i in range(n_nodes)}
    try:    clu = nx.clustering(G, weight="weight")
    except Exception: clu = {i:0.0 for i in range(n_nodes)}
    try:    pr = nx.pagerank(G, weight="weight") if G.number_of_edges()>0 else {i:1.0/n_nodes for i in range(n_nodes)}
    except Exception: pr = {i:1.0/n_nodes for i in range(n_nodes)}
    f = np.zeros((n_nodes,4))
    for i in range(n_nodes):
        f[i,0]=btw.get(i,0.0); f[i,1]=eig.get(i,0.0); f[i,2]=clu.get(i,0.0); f[i,3]=pr.get(i,0.0)
    return f

dataset = torch.load(PT_IN, weights_only=False)
n_nodes = dataset[0].x.shape[0]
print("Graphes:", len(dataset), "| noeuds:", n_nodes)

empty = 0
for k,d in enumerate(dataset):
    if d.edge_index.numel()==0: empty += 1
    topo = node_topo_features(d.edge_index, d.edge_attr, n_nodes)
    d.x = torch.tensor(np.hstack([d.x.numpy(), topo]), dtype=torch.float)
    if (k+1)%40==0: print(f"  {k+1}/{len(dataset)}")

print("Graphes vides:", empty)
print("Nouvelle dim features noeud:", dataset[0].x.shape[1], "(était 2)")
print("  [0]ChaCo [1]degré [2]betweenness [3]eigenvector [4]clustering [5]pagerank")
torch.save(dataset, PT_TOPO)
print("Sauvegardé:", PT_TOPO)

## 5. Chargement des variables cliniques (fusion) et des labels
18 variables d'admission (sans fuite). Alignées sur l'ordre des graphes.

In [ ]:
import re, csv
CLIN_NUM = ["Age","Glucose","Leucocytes","CRP","INR","NIHSS at admission","mRS premorbid"]
CLIN_BIN = ["Atrial fibrillation","Hypertension","Diabetes","Hyperlipidemia",
            "Anticoagulation","Lipid lowering drugs","PAIs","Wake-up","In-House","Referral"]
PID_RE = re.compile(r"(sub-stroke\d+)", re.IGNORECASE)

def load_clinical(dataset, mni_dir):
    rows = {}
    for name in os.listdir(mni_dir):
        d = os.path.join(mni_dir, name)
        if not os.path.isdir(d): continue
        m = PID_RE.search(name)
        if not m: continue
        pid = m.group(1).lower()
        f = next((os.path.join(d,fn) for fn in os.listdir(d)
                  if fn.lower().endswith("demographic_baseline.csv")), None)
        if not f: continue
        with open(f, newline="", encoding="utf-8-sig") as fh:
            r = csv.DictReader(fh); r.fieldnames=[h.strip() for h in r.fieldnames]
            row = next(r, None)
            if row: rows[pid] = {k.strip():(v.strip() if v else "") for k,v in row.items()}
    def tf(x):
        try: return float(x)
        except: return np.nan
    for d in dataset:
        rr = rows.get(d.pid.lower(), {})
        vals = [tf(rr.get(c,"")) for c in CLIN_NUM+CLIN_BIN]
        sx = (rr.get("Sex","") or "").upper()
        vals.append(1.0 if sx=="M" else (0.0 if sx=="F" else np.nan))
        d.clinical_raw = torch.tensor(vals, dtype=torch.float).unsqueeze(0)
    return dataset

dataset = load_clinical(dataset, BASE)
y = np.array([int(d.y) for d in dataset])
print("Labels:", len(y), "| classe1 (mRS>2):", int(y.sum()), "| classe0:", len(y)-int(y.sum()))

## 6. Modèle GAT (features de nœuds = 6)
Même architecture que ton bloc 1, mais `in_channels=6` pour absorber la topologie.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

class StrokeGATTopo(nn.Module):
    def __init__(self, in_channels=6, hidden=64, heads=4, clinical_dim=18,
                 clinical_proj=16, dropout=0.4, use_clinical=True):
        super().__init__()
        self.use_clinical=use_clinical; self.dropout=dropout
        self.gat1=GATConv(in_channels, hidden, heads=heads, dropout=dropout, add_self_loops=True)
        self.gat2=GATConv(hidden*heads, hidden, heads=1, concat=False, dropout=dropout, add_self_loops=True)
        gdim=hidden*2
        if use_clinical:
            self.clin=nn.Sequential(nn.Linear(clinical_dim,clinical_proj),nn.ELU(),nn.Dropout(dropout))
            head_in=gdim+clinical_proj
        else:
            self.clin=None; head_in=gdim
        self.head=nn.Sequential(nn.Linear(head_in,64),nn.LayerNorm(64),nn.ELU(),
                                nn.Dropout(dropout),nn.Linear(64,1))
    def forward(self, data):
        x,ei,b = data.x, data.edge_index, data.batch
        x=F.elu(self.gat1(x,ei)); x=F.dropout(x,p=self.dropout,training=self.training)
        x=F.elu(self.gat2(x,ei))
        g=torch.cat([global_mean_pool(x,b), global_max_pool(x,b)],dim=1)
        if self.use_clinical: g=torch.cat([g, self.clin(data.clinical)],dim=1)
        return self.head(g).squeeze(-1)
print("Modèle StrokeGATTopo défini (in_channels=6).")

## 7. Entraînement CV 5×10 + prédictions out-of-fold
On entraîne deux variantes : **GAT-topo graphe-seul** et **GAT-topo + clinique**.
Normalisation (features nœuds + clinique) fit sur le train de chaque fold.

In [ ]:
import copy
from torch_geometric.loader import DataLoader
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS,N_REPEATS,SEED = 5,10,42
MAX_EPOCHS,PATIENCE,LR,WD,BS,VAL = 200,20,5e-3,5e-4,16,0.2

def norm_fit(dataset, idx):
    # normalise les 6 features de noeuds + la clinique, sur le train
    Xn = np.vstack([dataset[i].x.numpy() for i in idx])
    nm, ns = Xn.mean(0), Xn.std(0); ns=np.where(ns<1e-8,1.0,ns)
    Cn = np.vstack([dataset[i].clinical_raw.numpy() for i in idx])
    cmed = np.nanmedian(Cn,0); cmed=np.where(np.isnan(cmed),0.0,cmed)
    Ci = np.where(np.isnan(Cn),cmed,Cn); cm,cs=Ci.mean(0),Ci.std(0); cs=np.where(cs<1e-8,1.0,cs)
    for d in dataset:
        d.x = torch.tensor((d.x.numpy()-nm)/ns, dtype=torch.float)
        cv = d.clinical_raw.numpy().copy(); cv=np.where(np.isnan(cv),cmed,cv)
        d.clinical = torch.tensor((cv-cm)/cs, dtype=torch.float)

def run_cv(use_clinical, tag):
    rskf=RepeatedStratifiedKFold(n_splits=N_SPLITS,n_repeats=N_REPEATS,random_state=SEED)
    oof=np.zeros(len(y)); cnt=np.zeros(len(y)); fold_aucs=[]
    Xd=np.zeros((len(y),1))
    for k,(tr,te) in enumerate(rskf.split(Xd,y)):
        # recharge propre à chaque fold (features brutes) puis normalise sur train
        ds=torch.load(PT_TOPO, weights_only=False); load_clinical(ds, BASE)
        seed=SEED+k; ytr=y[tr]
        tin,val=train_test_split(tr,test_size=VAL,stratify=ytr,random_state=seed)
        norm_fit(ds, tin)
        tl=DataLoader([ds[i] for i in tin],batch_size=BS,shuffle=True)
        vl=DataLoader([ds[i] for i in val],batch_size=64)
        el=DataLoader([ds[i] for i in te],batch_size=64)
        n1=int(ytr.sum()); n0=len(ytr)-n1
        pw=torch.tensor([n0/max(n1,1)],dtype=torch.float,device=DEVICE)
        crit=nn.BCEWithLogitsLoss(pos_weight=pw)
        torch.manual_seed(seed)
        model=StrokeGATTopo(in_channels=6,use_clinical=use_clinical).to(DEVICE)
        opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
        best,bs_state,wait=-1,None,0
        for _ in range(MAX_EPOCHS):
            model.train()
            for batch in tl:
                batch=batch.to(DEVICE); opt.zero_grad()
                loss=crit(model(batch),batch.y.float()); loss.backward(); opt.step()
            model.eval(); ps=[];ys=[]
            with torch.no_grad():
                for batch in vl:
                    batch=batch.to(DEVICE)
                    ps.append(torch.sigmoid(model(batch)).cpu().numpy()); ys.append(batch.y.cpu().numpy())
            ps,ys=np.concatenate(ps),np.concatenate(ys)
            va=roc_auc_score(ys,ps) if len(np.unique(ys))>1 else 0.5
            if va>best: best,bs_state,wait=va,copy.deepcopy(model.state_dict()),0
            else:
                wait+=1
                if wait>=PATIENCE: break
        if bs_state: model.load_state_dict(bs_state)
        model.eval(); ps=[]
        with torch.no_grad():
            for batch in el:
                batch=batch.to(DEVICE); ps.append(torch.sigmoid(model(batch)).cpu().numpy())
        ps=np.concatenate(ps)
        oof[te]+=ps; cnt[te]+=1
        fold_aucs.append(roc_auc_score(y[te],ps) if len(np.unique(y[te]))>1 else 0.5)
        if (k+1)%N_SPLITS==0:
            print(f"  {tag} rép {(k+1)//N_SPLITS}/{N_REPEATS}  AUC={np.mean(fold_aucs[-N_SPLITS:]):.3f}")
    oof/=np.maximum(cnt,1)
    print(f"=> {tag}: AUC {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")
    return oof

import warnings; warnings.filterwarnings("ignore")
print(">>> GAT-topo graphe-seul"); oof_topo_graph = run_cv(False,"topo-graphe")
print(">>> GAT-topo + clinique");  oof_topo_clin  = run_cv(True ,"topo+clin")

## 8. Baselines out-of-fold (mêmes splits) pour la comparaison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
try:
    from xgboost import XGBClassifier; HAS_XGB=True
except: HAS_XGB=False

# reconstruire X_chaco (vecteur plat) et X_clin alignés
ds0=torch.load(PT_IN, weights_only=False)  # x d'origine (ChaCo en colonne 0)
X_chaco=np.vstack([d.x[:,0].numpy() for d in ds0])
load_clinical(dataset, BASE)
X_clin=np.vstack([d.clinical_raw.numpy() for d in dataset])
X_cc=np.hstack([X_chaco,X_clin])

def mk_lr(): return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),
                              ("clf",LogisticRegression(max_iter=2000,class_weight="balanced"))])
def mk_xgb():
    n1=int(y.sum()); n0=len(y)-n1
    clf=XGBClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,subsample=0.8,
                      colsample_bytree=0.8,eval_metric="logloss",random_state=SEED,
                      scale_pos_weight=n0/max(n1,1)) if HAS_XGB else None
    return Pipeline([("imp",SimpleImputer(strategy="median")),("clf",clf)])

def oof_pred(X, mk):
    rskf=RepeatedStratifiedKFold(n_splits=N_SPLITS,n_repeats=N_REPEATS,random_state=SEED)
    acc=np.zeros(len(y)); cnt=np.zeros(len(y))
    for tr,te in rskf.split(X,y):
        m=mk(); m.fit(X[tr],y[tr]); acc[te]+=m.predict_proba(X[te])[:,1]; cnt[te]+=1
    return acc/np.maximum(cnt,1)

print("Baselines OOF...")
oof_chaco = oof_pred(X_chaco, mk_xgb if HAS_XGB else mk_lr)   # ChaCo vecteur
oof_b4    = oof_pred(X_cc,    mk_xgb if HAS_XGB else mk_lr)   # ChaCo+clin
print("AUC ChaCo-vecteur:", round(roc_auc_score(y,oof_chaco),3),
      "| AUC ChaCo+clin:", round(roc_auc_score(y,oof_b4),3))

## 9. Test de DeLong — la topologie apporte-t-elle enfin quelque chose ?

- **C1-topo** : GAT-topo graphe-seul vs ChaCo-vecteur → *la topologie explicite aide-t-elle ?*
- **C2-topo** : GAT-topo + clinique vs ChaCo+clinique → *le modèle complet gagne-t-il ?*

In [ ]:
from scipy import stats
def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N); T2[J]=T; return T2
def _fastdelong(pred, m):
    n=pred.shape[1]-m; pos=pred[:,:m]; neg=pred[:,m:]; k=pred.shape[0]
    tx=np.empty((k,m)); ty=np.empty((k,n)); tz=np.empty((k,m+n))
    for r in range(k):
        tx[r]=_midrank(pos[r]); ty[r]=_midrank(neg[r]); tz[r]=_midrank(pred[r])
    aucs=tz[:,:m].sum(1)/m/n-(m+1.0)/2.0/n
    v01=(tz[:,:m]-tx)/n; v10=1.0-(tz[:,m:]-ty)/m
    cov=np.cov(v01)/m+np.cov(v10)/n; return aucs,cov
def delong(y_true,s1,s2):
    o=(-y_true).argsort(kind="mergesort"); m=int(y_true.sum())
    pred=np.vstack((s1,s2))[:,o]; aucs,cov=_fastdelong(pred,m)
    diff=aucs[0]-aucs[1]; var=cov[0,0]+cov[1,1]-2*cov[0,1]
    se=np.sqrt(var) if var>0 else 1e-12; z=diff/se; p=2*(1-stats.norm.cdf(abs(z)))
    return float(aucs[0]),float(aucs[1]),float(diff),float(p),(diff-1.96*se,diff+1.96*se)

tests=[("C1-topo  GAT-topo-graphe vs ChaCo-vecteur", oof_topo_graph, oof_chaco),
       ("C2-topo  GAT-topo+clin   vs ChaCo+clin",     oof_topo_clin,  oof_b4)]
print("="*70)
rows=[]
for name,s1,s2 in tests:
    a1,a2,d,p,ci=delong(y,s1,s2)
    verdict = "SIGNIFICATIF" if p<0.05 else "non significatif"
    sens = "en faveur du GAT-topo" if d>0 else "en faveur de la baseline"
    print(f"\n{name}")
    print(f"   AUC GAT={a1:.3f}  AUC base={a2:.3f}  Δ={d:+.3f}  p={p:.3f}  [{verdict}, {sens}]")
    rows.append([name,f"{a1:.4f}",f"{a2:.4f}",f"{d:+.4f}",f"{p:.4f}",verdict,sens])

import csv as _csv
with open(OUT_CSV,"w",newline="",encoding="utf-8") as f:
    w=_csv.writer(f); w.writerow(["comparaison","AUC_GAT","AUC_base","delta","p","verdict","sens"]); w.writerows(rows)
print("\nCSV:", OUT_CSV)
print("="*70)
print("LECTURE :")
print("  C1-topo significatif & en faveur du GAT-topo => la topologie AIDE")
print("     -> contribution CS POSITIVE.")
print("  C1-topo non significatif => même enrichie, la topologie n'aide pas")
print("     -> résultat négatif RENFORCÉ (connectome normatif en cause).")

---
### Interprétation du résultat

- **Si C1-topo est significatif en faveur du GAT-topo** → tu tiens une **contribution CS positive** :
  une représentation topologiquement enrichie exploite la structure du disconnectome.
- **Sinon** → **résultat négatif renforcé**, publiable et scientifiquement plus fort :
  même en injectant explicitement la topologie, le disconnectome normatif ne fournit pas
  d'information structurelle exploitable.

⚠️ Ne relance pas en boucle en changeant les paramètres pour « forcer » p<0.05 (p-hacking).
Un run propre, un verdict.